# 配套实践 05-02：从指令预测任务槽位

本练习用组合生成的机器人指令训练小型语言编码器，同时预测动作、颜色、对象和目标。模型刻意保持简单，用于检查表示是否保留任务信息，而不是模拟完整大语言模型。依赖：PyTorch；CPU 即可运行。

<a href="https://qi-robotics.github.io/robot-world-model-tutorial/basics/language-encoding/" target="_blank">返回课程正文</a>

In [ ]:
import itertools  # 生成动作、颜色、对象和目标的组合
import torch  # 训练小型 embedding 编码器和分类头
torch.manual_seed(12)  # 固定参数初始化和训练顺序

## 1. 生成组合指令

保留两个组合只用于测试，观察模型能否把训练中见过的槽位重新组合。指令使用固定模板，因此实验只验证基础接口，不代表解决了自然语言理解。

In [ ]:
actions = ["拿起", "推动"]  # 定义两种动作类别
colors = ["红色", "蓝色"]  # 定义两种颜色属性
objects = ["杯子", "方块"]  # 定义两种操作对象
targets = ["托盘", "左侧"]  # 定义两种目标位置
examples = [(f"{action} {color} {obj} 到 {target}", action, color, obj, target) for action, color, obj, target in itertools.product(actions, colors, objects, targets)]  # 生成全部槽位组合
held_out_texts = {"推动 红色 杯子 到 托盘", "拿起 蓝色 方块 到 左侧"}  # 指定两个未见组合
train_examples = [example for example in examples if example[0] not in held_out_texts]  # 构造不含保留组合的训练集
test_examples = [example for example in examples if example[0] in held_out_texts]  # 构造组合泛化测试集
vocab_words = sorted({word for example in examples for word in example[0].split()})  # 从全部基础词建立教学词表
vocab = {word: index + 1 for index, word in enumerate(vocab_words)}  # 从一开始编号并为 PAD 保留零
vocab["[PAD]"] = 0  # 将补齐 token 固定为零
label_maps = [{value: index for index, value in enumerate(values)} for values in (actions, colors, objects, targets)]  # 为四个分类任务建立标签编号
print("训练样本：", len(train_examples), "测试组合：", len(test_examples))  # 显示数据划分规模

In [ ]:
def make_batch(rows):  # 把若干文本与槽位标签整理为规则张量
    sequences = [[vocab[word] for word in row[0].split()] for row in rows]  # 把空格分隔的词转换为 id 序列
    length = max(len(sequence) for sequence in sequences)  # 读取当前 batch 的最大长度
    ids = torch.zeros((len(rows), length), dtype=torch.long)  # 创建以 PAD 填充的 token 张量
    mask = torch.zeros((len(rows), length), dtype=torch.float32)  # 创建有效 token 遮罩
    for index, sequence in enumerate(sequences):  # 逐条写入变长序列
        ids[index, :len(sequence)] = torch.tensor(sequence)  # 写入真实 token id
        mask[index, :len(sequence)] = 1.0  # 标记真实 token 位置
    labels = [torch.tensor([label_maps[slot][row[slot + 1]] for row in rows]) for slot in range(4)]  # 分别创建四组分类标签
    return ids, mask, labels  # 返回模型输入和监督目标
train_ids, train_mask, train_labels = make_batch(train_examples)  # 整理训练张量
test_ids, test_mask, test_labels = make_batch(test_examples)  # 整理测试张量
assert train_ids.shape == train_mask.shape  # 检查 mask 与 token 完全对齐

## 2. masked pooling 与多任务槽位头

所有槽位共享同一个句子表示，每个分类头只读取自己负责的标签。这个结构能检查任务信息是否可读出，但因为平均池化忽略词序，仍有明确能力边界。

In [ ]:
class SlotEncoder(torch.nn.Module):  # 定义共享语言表示和四个槽位分类头
    def __init__(self, vocab_size, hidden_size=24):  # 接收词表大小和表示维度
        super().__init__()  # 初始化神经网络基类
        self.embedding = torch.nn.Embedding(vocab_size, hidden_size, padding_idx=0)  # 创建忽略 PAD 的 token embedding
        self.heads = torch.nn.ModuleList([torch.nn.Linear(hidden_size, 2) for _ in range(4)])  # 为四个二分类槽位创建独立读出头
    def forward(self, ids, mask):  # 定义从 token 到四组 logits 的前向传播
        features = self.embedding(ids)  # 查表得到 [B,L,D] token 表示
        weights = mask.unsqueeze(-1)  # 扩展 mask 以匹配表示维度
        pooled = (features * weights).sum(dim=1) / weights.sum(dim=1).clamp_min(1.0)  # 只汇总真实 token
        return [head(pooled) for head in self.heads], pooled  # 返回四组分类分数和共享任务向量
model = SlotEncoder(len(vocab))  # 创建小型语言编码器
optimizer = torch.optim.Adam(model.parameters(), lr=0.04)  # 使用 Adam 更新 embedding 与分类头
for step in range(201):  # 在小型组合数据上进行固定步数训练
    logits, _ = model(train_ids, train_mask)  # 计算训练样本的四组预测
    loss = sum(torch.nn.functional.cross_entropy(prediction, target) for prediction, target in zip(logits, train_labels))  # 汇总四个槽位的交叉熵
    optimizer.zero_grad()  # 清除上一步累积梯度
    loss.backward()  # 反向传播计算所有参数梯度
    optimizer.step()  # 根据梯度更新参数
    if step in (0, 50, 100, 200):  # 在少量关键步打印学习进度
        print(f"step={step:03d}, loss={loss.item():.4f}")  # 输出当前总损失

In [ ]:
model.eval()  # 切换到评估模式以表达当前阶段不再训练
with torch.no_grad():  # 关闭梯度记录以减少评估开销
    predictions, task_vectors = model(test_ids, test_mask)  # 对未见组合进行槽位预测
    predicted_labels = [prediction.argmax(dim=1) for prediction in predictions]  # 把每个分类头的分数转换成类别编号
    correct = sum((prediction == target).sum().item() for prediction, target in zip(predicted_labels, test_labels))  # 统计全部槽位预测正确数
    total = sum(target.numel() for target in test_labels)  # 统计全部待预测槽位数
print(f"未见组合槽位准确率: {correct}/{total} = {correct / total:.1%}")  # 报告组合测试结果
print("任务向量形状：", tuple(task_vectors.shape))  # 显示句子级表示接口
assert task_vectors.shape == (len(test_examples), 24)  # 验证任务向量形状

## 3. 主动观察失败边界

当前词表和标签都没有“不要”这个否定槽位。即使模型能正确重组动作、颜色和对象，也不能据此声称它理解否定、指代或开放世界词汇。尝试加入 `不要 推动 红色 杯子 到 托盘`，思考应新增“禁止动作”标签，还是让系统进入澄清/拒绝分支。